# 07 Monitoring simulation

Purpose: score monthly out-of-time batches with the frozen champion, and check each
batch for input drift, score drift, calibration and ranking, with traffic lights.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
T = ROOT / "reports" / "tables"; M = ROOT / "models"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Batch results

In [ ]:
batches = pd.read_csv(T / "monitoring_batches.csv")
display(batches[["month", "loans", "actual_bad_rate", "predicted_pd", "actual_to_predicted",
                 "calibration_light", "psi_score", "psi_score_light", "batch_gini", "gini_light",
                 "overall", "retrain_review"]].round(4))

## 2. Summary

In [ ]:
display(pd.Series(json.loads((M / "run_summary.json").read_text())).to_frame("value"))

Calibration is the dominant signal: predicted PD is below the observed rate in most
out-of-time months, which is the drift identified in the validation report. The
retraining review rule (two consecutive red batches) is applied in the table.

## 3. Feature drift by month

In [ ]:
feats = pd.read_csv(T / "monitoring_features.csv")
display(feats.pivot(index="feature", columns="month", values="psi").round(3))

Features with PSI above 0.25 in any month need a root-cause review, as described in the validation report.